# **Dhruv's Contribution**

Installing dependencies

In [ ]:
!pip install langchain faiss-cpu sentence-transformers transformers datasets
!pip install huggingface_hub openai dspy pandas
!pip install -U langchain-huggingface langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 36.8 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstallin

Importing Libraries

In [ ]:
#Imports
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain.llms import HuggingFacePipeline
from langchain.chains import RetrievalQA
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS as LC_FAISS
import faiss

In [ ]:
#Environment Fix
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

Data Loading

In [ ]:
#Load Biomedical Data
df = pd.read_parquet("hf://datasets/rag-datasets/rag-mini-bioasq/data/passages.parquet/part.0.parquet")
df_test = pd.read_parquet("hf://datasets/rag-datasets/rag-mini-bioasq/data/test.parquet/part.0.parquet")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
#Embeddings
embedder = SentenceTransformer("all-MiniLM-L6-v2", device='cpu')
passages = df['passage'].sample(n=5000, random_state=42).tolist()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
#FAISS Index
passage_embeddings = []
batch_size = 32
for i in tqdm(range(0, len(passages), batch_size)):
    batch = passages[i:i + batch_size]
    emb = embedder.encode(batch, convert_to_numpy=True)
    passage_embeddings.append(emb)

passage_embeddings = np.vstack(passage_embeddings)
dimension = passage_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(passage_embeddings)

  0%|          | 0/157 [00:00<?, ?it/s]/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
100%|██████████| 157/157 [11:35<00:00,  4.43s/it]


In [ ]:
#LangChain-compatible Retriever
embedding_function = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
retriever_db = LC_FAISS.from_texts(passages, embedding_function)
retriever = retriever_db.as_retriever()

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from langchain_community.llms import HuggingFacePipeline

# Use a lightweight and open model
model_id = "google/flan-t5-base"

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

# Create HuggingFace pipeline
pipe = pipeline(
    "text2text-generation",  # Important: use text2text for T5
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
)

# Wrap in LangChain LLM interface
llm = HuggingFacePipeline(pipeline=pipe)

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Device set to use cpu
/tmp/ipython-input-4281328707.py:20: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=pipe)


In [ ]:
#RetrievalQA Chain
qa_chain = RetrievalQA.from_chain_type(llm=llm, retriever=retriever, return_source_documents=True)

In [ ]:
#Smart Context Formatter with Truncation + Fallback
def biomedical_chatbot(query):
    # Get relevant docs
    docs = retriever.get_relevant_documents(query)

    # If no docs are found, avoid hallucination
    if not docs or all(len(doc.page_content.strip()) == 0 for doc in docs):
        return "Sorry, I can only answer biomedical questions."

    # Use invoke to handle multiple output keys
    response = qa_chain.invoke(query)
    print("QA Response:", response)
    return response['result']


Chatbot Questions

In [ ]:
# Test the biomedical chatbot with different queries
questions = [
    "What are the symptoms of Parkinson's disease?",
    "How is type 2 diabetes managed?",
    "What causes chronic obstructive pulmonary disease (COPD)?",
    "What is the function of hemoglobin in the blood?",
    "What is the treatment for Crohn's disease?"
]

# Loop through each question and get the response from the chatbot
for question in questions:
    print(f"Question: {question}")
    print(f"Answer: {biomedical_chatbot(question)}\n")


/tmp/ipython-input-681379986.py:4: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  docs = retriever.get_relevant_documents(query)
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
Token indices sequence length is longer than the specified maximum sequence length for this model (1218 > 512). Running this sequence through the model will result in indexing errors


Question: What are the symptoms of Parkinson's disease?
QA Response: {'query': "What are the symptoms of Parkinson's disease?", 'result': 'severe and atypical progressive parkinsonism characterized by bilateral bradykinesia and rigidity, slowness of gait, postural instability, and falls, and poor or absent response to adequate levodopa treatment; (b) increased tendon reflexes associated or not with frank pyramidal signs, severe dysarthria, and less consistently, dysphagia, stridor, antecollis, and stimulus-sensitive myoclonus', 'source_documents': [Document(id='fae37314-1908-4463-9b0d-536a92e81483', metadata={}, page_content="INTRODUCTION: In addition to the classic triad (tremor, rigidity and akinesia), \nParkinson's disease (PD) is also accompanied by several non-motor disorders.\nAIM: To carry out an updated review of these non-motor symptoms in terms of \ntheir pathophysiology, epidemiology, clinical features and treatment.\nDEVELOPMENT: Autonomic disorders such as seborrhoeic derm

/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


QA Response: {'query': 'How is type 2 diabetes managed?', 'result': 'pharmacologic therapy of T2DM has been progressively enriched by several novel classes of drugs, trying to overcome these defects. The last, intriguing compounds come into the market are SGLT2 inhibitors, framing the kidney in a different scenario, not as site of a harmful disease complication, but rather as the means to correct hyperglycemia and fight the disease. Vildagliptin is a potent and selective inhibitor of dipeptidyl peptidase-IV (DPP-4), orally active, that improves glycemic control in patients with type 2 diabetes (T2DM) primarily by enhancing pancreatic (alpha and beta) islet function. Thus vildagliptin has been shown both to improve insulin secretion and to suppress the inappropriate glucagon secretion seen in patients with T2DM. Vildagliptin reduces HbA(1c) when given as monotherapy, without weight gain and with minimal hypoglycemia, or in combination with the most commonly prescribed classes of oral hy

/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


QA Response: {'query': 'What causes chronic obstructive pulmonary disease (COPD)?', 'result': 'Cigarette smoking', 'source_documents': [Document(id='900012bd-375d-443a-a233-1390a328af3d', metadata={}, page_content='AIM: Osteoporosis is a frequent comorbidity in patients with chronic obstructive \npulmonary disease (COPD). We have studied the risk of major osteoporotic \nfracture and hip fracture in patients with COPD.\nPATIENTS AND METHODS: A multicenter cross-sectional study was performed in Spain \nin 26 hospitals of 16 regional communities. Patients diagnosed with COPD who \nrequired admission to the Internal Medicine Service due to exacerbation of their \nrespiratory disease were enrolled. COPD was confirmed by post-bronchodilator \nspirometry in stable state: maximum expiratory volume in the first second (FEV₁) \n< 80% of the theoretical value and quotient FEV(1)/FVC < 0.70 and percent \npredicted after the administration of a bronchodilator. Dyspnea was evaluated \nwith the modif

/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


QA Response: {'query': 'What is the function of hemoglobin in the blood?', 'result': 'hemoglobin accumulation during human erythropoiesis', 'source_documents': [Document(id='a7e40c84-2d0c-4451-b292-dee143809074', metadata={}, page_content='The design and evaluation of therapies for the sickle cell and beta-thalassemia \nsyndromes rely on our understanding of hemoglobin accumulation during human \nerythropoiesis. Here we report direct measurements of hemoglobin composition and \nmessenger RNA (mRNA) levels in cultured CD34(+) cells and correlate those \nmeasurements with studies of freshly obtained bone marrow samples. Hemoglobin \nlevels in differentiating cells were also compared with morphologic, \nimmunophenotypic, and cell cycle assessments. A population of large \npreproerythroblasts was first identified within 24 hours and became the dominant \npopulation by day 5. The transition from proerythroblast to basophilic \nnormoblast occurred later, from days 7 to 9, and correlated with

/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


QA Response: {'query': "What is the treatment for Crohn's disease?", 'result': 'metronidazole', 'source_documents': [Document(id='136dff32-b95a-443d-9b33-a2b8b64402c8', metadata={}, page_content='Recently, several medical treatments for ulcerative colitis (UC) have been \ndeveloped, including 5-aminosalicylic acids (5-ASAs), corticosteroids, \nthiopurine, calcineurin inhibitors, and anti-tumor necrosis factor (TNF) α \ntreatments. Treatment options including calcineurin inhibitors and anti-TNF \ntreatment for refractory UC are discussed in this article. Furthermore, upcoming \ntreatments are introduced, such as golimumab, vedolizumab, AJM300, tofacitinib. \nBudesonide foamwill be used as one treatment option in patients with distal \ncolitis. Herbal medicine, such as Qing-Dai is also effective for active UC and \nmay be useful for patients who are refractory to anti-TNFα treatments. In the \nnear future, physicians will able to use many different treatments for UC \npatients. However, 

Evaluation Metrics

In [ ]:
!pip install rouge-score bert-score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.5 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=d59f0369188f9316f7ac1e3137c95f9de711e650c84662ea35fe89c84f507b17
  Stored in directory: /root/.cache/pip/wheels/1e/19/43/8a442dc83660ca25e163e1bd1f89919284ab0d0c1475475148
Successfully built rouge-score


In [ ]:
from tqdm import tqdm
from rouge_score import rouge_scorer
import bert_score

def evaluate_chatbot(chatbot, questions, reference_answers, limit=None):
    """
    Evaluates the chatbot using ROUGE-L and BERTScore.
    Optionally limits to first `limit` examples for faster testing.
    """
    # Optionally limit the number of questions to evaluate
    if limit:
        questions = questions[:limit]
        reference_answers = reference_answers[:limit]

    generated_answers = []

    # Generate responses for the provided questions
    print("Generating responses...")
    for q in tqdm(questions, desc="Generating responses"):
        response = chatbot(q)  # Get chatbot's response
        generated_answers.append(response)

    # ROUGE-L Score calculation
    rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    rouge_scores = [rouge.score(ref, pred)['rougeL'].fmeasure for ref, pred in zip(reference_answers, generated_answers)]
    avg_rouge_l = sum(rouge_scores) / len(rouge_scores)

    # BERTScore calculation
    P, R, F1 = bert_score.score(generated_answers, reference_answers, lang='en', verbose=False)
    avg_bertscore = F1.mean().item()

    # Print only the ROUGE and BERT scores
    print("=" * 50)
    print("Evaluation Results")
    print("=" * 50)
    print(f"Average ROUGE-L Score : {avg_rouge_l:.4f}")
    print(f"Average BERTScore F1  : {avg_bertscore:.4f}")
    print("=" * 50)

    return generated_answers

In [ ]:
# Assuming you have a DataFrame `df_test` containing the test data
questions = df_test['question'].tolist()  # List of questions
reference_answers = df_test['answer'].tolist()  # List of reference answers

# Run evaluation on a subset of 5 questions for testing
generated_responses = evaluate_chatbot(biomedical_chatbot, questions, reference_answers, limit=5)

Generating responses...


Generating responses:   0%|          | 0/5 [00:00<?, ?it/s]/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
Generating responses:  20%|██        | 1/5 [00:26<01:44, 26.06s/it]

QA Response: {'query': 'Is Hirschsprung disease a mendelian or a multifactorial disorder?', 'result': 'multifactorial', 'source_documents': [Document(id='f02d727a-37eb-43d4-b1d2-b22ec05e6e3b', metadata={}, page_content='Hirschsprung disease is a congenital form of aganglionic megacolon that results \nfrom cristopathy. Hirschsprung disease usually occurs as a sporadic disease, \nalthough it may be associated with several inherited conditions, such as \nmultiple endocrine neoplasia type 2. The rearranged during transfection (RET) \nproto-oncogene is the major susceptibility gene for Hirschsprung disease, and \ngermline mutations in RET have been reported in up to 50% of the inherited forms \nof Hirschsprung disease and in 15-20% of sporadic cases of Hirschsprung disease. \nThe prevalence of Hirschsprung disease in multiple endocrine neoplasia type 2 \ncases was recently determined to be 7.5% and the cooccurrence of Hirschsprung \ndisease and multiple endocrine neoplasia type 2 has been r

/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
Generating responses:  40%|████      | 2/5 [01:46<02:54, 58.09s/it]

QA Response: {'query': 'List signaling molecules (ligands) that interact with the receptor EGFR?', 'result': 'seven ligands are known to bind EGFR with affinities ranging from sub-nanomolar to near micromolar dissociation constants. In the case of EGFR, distinct conformational states assumed upon binding a ligand is thought to be a determining factor in activation of a downstream signaling network.', 'source_documents': [Document(id='26d120b9-7e64-4886-9773-687fd5663e88', metadata={}, page_content='The epidermal growth factor receptor (EGFR) is a member of the receptor tyrosine \nkinase family that plays a role in multiple cellular processes. Activation of \nEGFR requires binding of a ligand on the extracellular domain to promote \nconformational changes leading to dimerization and transphosphorylation of \nintracellular kinase domains. Seven ligands are known to bind EGFR with \naffinities ranging from sub-nanomolar to near micromolar dissociation constants. \nIn the case of EGFR, dis

...

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `RobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


Evaluation Results
Average ROUGE-L Score : 0.0397
Average BERTScore F1  : 0.7962


In [ ]:
import torch

# Save the model's state_dict (weights)
model_save_path = 'rag_model.pth'
torch.save(model.state_dict(), model_save_path)
print(f"Model saved to {model_save_path}")

# Save the tokenizer
tokenizer_save_path = 'rag_tokenizer'
tokenizer.save_pretrained(tokenizer_save_path)
print(f"Tokenizer saved to {tokenizer_save_path}")

Model saved to rag_model.pth
Tokenizer saved to rag_tokenizer


**LLM prompts:**

1. First Prompt: "How to install dependencies for a biomedical chatbot with LangChain, FAISS, and Hugging Face?"
2. Last Prompt: "How to speed up the LIME explainer for faster chatbot evaluation?"
